### Test automated pairwise alignments with PyMol
### Julian Moran
### 2026-02-20

In [3]:
import logging
import os
import requests
import subprocess
import tempfile
import time

from dotenv import load_dotenv
from pathlib import Path
from typing import Dict

import polars as pl

# Env
load_dotenv("../.env", override=True)
INSTALL_PATH = os.environ["INSTALL_PATH"]
API_URL_ALPHAFOLD_STRUCTPRED = os.environ["API_URL_ALPHAFOLD_STRUCTPRED"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [4]:
# ============================================================
#                           Args
# ============================================================

PM_ALIGN_METHOD = "cealign" # set to "align", "cealign", or "super"

args = {
    "data_file_protein_pairs": f"{INSTALL_PATH}/results/explore_aJain_iei_pipeline/defense_proteins_filt_n=297.tsv",
    "col_query_protein_ids": "defense_system_protein_id",
    "col_target_protein_ids": "protein_id",
    "out_dir": Path(f"{INSTALL_PATH}/results/test_pymol_align/"),
    "out_dir_af": Path(f"{INSTALL_PATH}/data_local/alphafold_structures"),
    "out_dir_pm": Path(f"{INSTALL_PATH}/data_local/pymol_alignments/{PM_ALIGN_METHOD}")
}

In [5]:
# ============================================================
#                           In
# ============================================================

args["out_dir"].mkdir(parents=True, exist_ok=True)
args["out_dir_pm"].mkdir(parents=True, exist_ok=True)
args["out_dir_af"].mkdir(parents=True, exist_ok=True)

df_protein_pairs = pl.read_csv(
    args["data_file_protein_pairs"],
    separator="\t",
    has_header=True
)

df_protein_pairs

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio,GRIID_gene
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64,str
"""WP_074585635.1""","""Eleos""","""Eleos""","""Bacillus mobilis,Bacillus cere…","""GCF_021655235_NZ_AP022971_Eleo…","""GCF_021655175.1_NZ_AP022953_00…","""GCF_021655235.1_NZ_AP022971_00…",14,189,"""A0A1Y6A5H7""","""A0A1I1GWW8""","""A0A1I1GWW8""","""A0A6Q8PGV8""",1.2200e-96,1,0.125,750,501,0.0,34,783,5,578,1.3890e-11,191,"""unreviewed""","""A0A6Q8PGV8_HUMAN""","""Mitofusin 2""","""MFN2""","""Homo sapiens (Human)""",null,null,"""MFN2""",null,"""UP000005640: Chromosome 1""","""mitochondrial fusion [GO:00080…","""mitochondrial outer membrane […","""mitochondrial outer membrane […","""GTP binding [GO:0005525]; GTPa…","""GO:0003924; GO:0005525; GO:000…","""cd09912; DLP_2; 1.;""","""1.20.5.110:FF:000012; Mitofusi…","""1.20.5.110; -; 1.;""3.40.50.300…","""IPR045063; Dynamin_N.;""IPR0068…","""PTHR10465:SF1; MITOFUSIN-2; 1.…","""PF00350; Dynamin_N; 1.;""PF0479…","""PS51718; G_DYNAMIN_2; 1.;""",null,null,"""34.0..783.0""",750,801,"""93..342""",100.0,33.33,93.63,"""['Dynamin-type G']""","""['ECO:0000259|PROSITE:PS51718'…","""5.0..578.0""",574,579,"""53..185""",100.0,23.17,99.14,"""['G']""","""['ECO:0000259|Pfam:PF01926']""",0.722846,574.0,0.991364,"""Yes"""
"""WP_074585635.1""","""Eleos""","""Eleos""","""Bacillus mobilis,Bacillus cere…","""GCF_021655235_NZ_AP022971_Eleo…","""GCF_021655175.1_NZ_AP022953_00…","""GCF_021655235.1_NZ_AP022971_00…",14,189,"""A0A556BFS8""","""A0A1I1GWW8""","""A0A1I1GWW8""","""A0A6Q8PFJ4""",1.2200e-96,1,0.128,696,494,0.0,40,735,7,574,2.5940e-12,198,"""unreviewed""","""A0A6Q8PFJ4_HUMAN""","""Mitofusin 2""","""MFN2""","""Homo sapiens (Human)""",null,null,"""MFN2""",null,"""UP000005640: Chromosome 1""","""mitochondrial fusion [GO:00080…","""mitochondrial outer membrane […","""mitochondrial outer membrane […","""GTP binding [GO:0005525]; GTPa…","""GO:0003924; GO:0005525; GO:000…","""cd09912; DLP_2; 1.;""","""3.40.50.300:FF:000214; Mitofus…","""1.20.5.110; -; 1.;""3.40.50.300…","""IPR045063; Dynamin_N.;""IPR0068…","""PTHR10465:SF1; MITOFUSIN-2; 1.…","""PF00350; Dynamin_N; 1.;""PF0479…","""PS51718; G_DYNAMIN_2; 1.;""",null,null,"""40.0..735.0""",696,808,"""93..342""",100.0,35.92,86.14,"""['Dynamin-type G']""","""['ECO:0000259|PROSITE:PS51718'…","""7.0..574.0""",568,579,"""352..545""",100.0,34.15,98.1,"""['Dynamin-like helical']""","""['ECO:0000259|Pfam:PF18709']""",0.716584,568.0,0.981002,"""Yes"""
"""WP_000434627.1""","""Eleos""","""Eleos""","""Escherichia coli""","""GCF_016776005_NZ_CP068823_Eleo…","""GCF_016775985.1_NZ_CP068827_02…","""GCF_020883255.1_NZ_CP086618_00…",27,176,"""Q9RFR9""","""A0A1I1GWW8""","""A0A1I1GWW8""","""O95140""",1.2200e-96,1,0.116,703,497,0.0,29,731,4,566,3.102

In [6]:
# ============================================================
#                      AlphaFold structures
# ============================================================

def get_alphafold_structure(
        uniprot_id: str,
        out_dir: Path,
        api_endpoint: str,
        wait_time: float = 1.0,
        overwrite: bool = False,
) -> str | None:
    
    # Check if file already exists:
    out_file = f"{out_dir}/{uniprot_id}_AF.pdb"
    if os.path.exists(out_file) and not overwrite:
        logger.warning(f"For {uniprot_id}, ALphaFold structure is already on file at {out_file}.")
        return out_file

    time.sleep(wait_time)
    url = f"{api_endpoint}/{uniprot_id}"
    logger.info(url)
    headers = {"accept": "application/json"}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.warning(f"For {uniprot_id}, failed to retrieve: {e}")
        return None
    url_pdb = response.json()[0].get("pdbUrl", None)

    if url_pdb is not None:
        logger.info(f"For {uniprot_id}, retrieving .pdb from {url_pdb}...")
        try:
            response = requests.get(url_pdb)
            response.raise_for_status()
        except requests.RequestException as e:
            logger.warning(f"For {uniprot_id}, failed to retrieve: {e}")
            return None

        with open(out_file, "wb") as f:
            f.write(response.content)
        logger.info(f"For {uniprot_id}, AlphaFold predicted structure written to {out_file}.")
        return out_file
    
    else:
        logger.warning(f"For {uniprot_id}, AlphaFold structure not found.")
        return None


def get_alphafold_structures(
    df_uniprot_ids: pl.DataFrame,
    col_query_uniprot_id: str,
    col_target_uniprot_id: str,
    out_dir: Path,
    api_endpoint: str,
    wait_time: float = 1.0,
    overwrite: bool = False,
) -> pl.DataFrame:
    
    # Initialize column structures
    col_query_structure_file_name = col_query_uniprot_id.replace("_id", "_AF_file")
    col_target_structure_file_name = col_target_uniprot_id.replace("_id", "_AF_file")
    col_query_structure_file, col_target_structure_file = [], []

    for i in range(len(df_uniprot_ids)):

        # Query proteins
        query_protein_structure_file = get_alphafold_structure(
            uniprot_id=df_uniprot_ids[i, col_query_uniprot_id],
            out_dir=out_dir,
            api_endpoint=api_endpoint,
            wait_time=wait_time,
            overwrite=overwrite
        )
        col_query_structure_file.append(query_protein_structure_file)

        # Target proteins
        target_protein_structure_file = get_alphafold_structure(
            uniprot_id=df_uniprot_ids[i, col_target_uniprot_id],
            out_dir=out_dir,
            api_endpoint=api_endpoint,
            wait_time=wait_time
        )
        col_target_structure_file.append(target_protein_structure_file)
    
    # Annotate file name to data
    df = df_uniprot_ids.with_columns(
        pl.Series(col_query_structure_file_name, col_query_structure_file)
    ).with_columns(
        pl.Series(col_target_structure_file_name, col_target_structure_file)
    )
    
    return df, col_query_structure_file_name, col_target_structure_file_name

df_structure_files = df_protein_pairs[args["col_query_protein_ids"], args["col_target_protein_ids"]]
df_structure_files, col_query_struct, col_target_struct = get_alphafold_structures(
    df_uniprot_ids=df_structure_files,
    col_query_uniprot_id=args["col_query_protein_ids"],
    col_target_uniprot_id=args["col_target_protein_ids"],
    out_dir=args["out_dir_af"],
    api_endpoint=API_URL_ALPHAFOLD_STRUCTPRED
)

df_structure_files

defense_system_protein_id,protein_id,defense_system_protein_AF_file,protein_AF_file
str,str,str,str
"""A0A1Y6A5H7""","""A0A6Q8PGV8""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
"""A0A556BFS8""","""A0A6Q8PFJ4""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
"""Q9RFR9""","""O95140""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
"""A0A556BFS8""","""A0A6Q8PGA3""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
"""Q9RFR9""","""A0A6Q8PGA3""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
…,…,…,…
"""Q3Y0J6""","""O43598""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
"""A0A829F2V1""","""O43598""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"
"""A0A6P1GFG5""","""Q17R31""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…"


In [20]:
# ============================================================
#                      PyMol: pairwise alignment
# ============================================================

def pymol_align_pair(
    pdb_query: str,
    pdb_target: str,
    out_path_structure: str,
    pymol_align_method: str,
    overwrite: bool = False,
) -> tuple[str, str]:
    
    # Check if alignment file already exists
    if os.path.exists(out_path_structure) and not overwrite:
        logger.warning(f"For {pdb_query}, {pdb_target} PyMol alignment is already on file at {out_path_structure}. Add overwrite=True if you wish to run alignment again.")
        return None, None

    pymol_script = f"""
from pymol import cmd

cmd.load(r"{pdb_query}", "s1")
cmd.load(r"{pdb_target}", "s2")

cmd.hide("everything")
cmd.show("cartoon", "s1")
cmd.show("cartoon", "s2")
cmd.color("blue", "s1")
cmd.color("red", "s2")

r = cmd.{pymol_align_method}("s1", "s2")

print("METRICS\\t%f\\t%d" % (r[0], r[1]))
cmd.save(r"{out_path_structure}")
cmd.quit()
"""

    # Run alignment
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as tmp:
        tmp.write(pymol_script)
        script_path = tmp.name
    result = subprocess.run(
        ["pymol", "-cq", script_path],
        capture_output=True,
        text=True,
        check=True,
    )

    logger.info(f"For {pdb_query},{pdb_target}, wrote alignment results to {out_path_structure}.")

    # Capture alignment performance metrics
    rmsd = None
    aligned_atoms = None

    for line in result.stdout.splitlines():
        if line.startswith("METRICS"):
            _, rmsd, aligned_atoms = line.split("\t")
            rmsd = float(rmsd)
            aligned_atoms = int(aligned_atoms)
    return rmsd, aligned_atoms


def pymol_align_pairs(
    df_structure_files: pl.DataFrame,
    col_query_id: str,
    col_query_struct: str,
    col_target_id: str,
    col_target_struct: str,
    out_dir: str,
    pymol_align_method: str,
    overwrite: bool = False,
) -> pl.DataFrame:
    
    col_struct_paths, col_rmsd, col_aligned_atoms = [], [], []

    for i in range(len(df_structure_files)):
        query_id = df_structure_files[i, col_query_id]
        target_id = df_structure_files[i, col_target_id]
        out_path_structure = f"{out_dir}/{query_id}_{target_id}_aligned.pse"

        rmsd, aligned_atoms = pymol_align_pair(
            pdb_query=df_structure_files[i, col_query_struct],
            pdb_target=df_structure_files[i, col_target_struct],
            out_path_structure=out_path_structure,
            pymol_align_method=pymol_align_method,
            overwrite=overwrite
        )
        col_struct_paths.append(out_path_structure)
        col_rmsd.append(rmsd)
        col_aligned_atoms.append(aligned_atoms)
    
    df = df_structure_files.with_columns(
        pl.Series("PyMOL_aligned_filePath", col_struct_paths)
    ).with_columns(
        pl.Series("PyMOL_RMSD", col_rmsd)
    ).with_columns(
        pl.Series("PyMOL_aligned_atoms", col_aligned_atoms)
    )
    return df

df_structures_aligned = pymol_align_pairs(
    df_structure_files=df_structure_files,
    col_query_id=args["col_query_protein_ids"],
    col_query_struct=col_query_struct,
    col_target_id=args["col_target_protein_ids"],
    col_target_struct=col_target_struct,
    pymol_align_method=PM_ALIGN_METHOD,
    out_dir=args["out_dir_pm"]
)

df_structures_aligned

INFO:__main__:For /home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/A0A1Y6A5H7_AF.pdb,/home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/A0A6Q8PGV8_AF.pdb, wrote alignment results to /home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/pymol_alignments/cealign/A0A1Y6A5H7_A0A6Q8PGV8_aligned.pse.
INFO:__main__:For /home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/A0A556BFS8_AF.pdb,/home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/A0A6Q8PFJ4_AF.pdb, wrote alignment results to /home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/pymol_alignments/cealign/A0A556BFS8_A0A6Q8PFJ4_aligned.pse.
INFO:__main__:For /home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/Q9RFR9_AF.pdb,/home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/O95140_AF.pdb, wrote alignment

defense_system_protein_id,protein_id,defense_system_protein_AF_file,protein_AF_file,PyMOL_aligned_filePath,PyMOL_RMSD,PyMOL_aligned_atoms
str,str,str,str,str,null,null
"""A0A1Y6A5H7""","""A0A6Q8PGV8""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
"""A0A556BFS8""","""A0A6Q8PFJ4""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
"""Q9RFR9""","""O95140""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
"""A0A556BFS8""","""A0A6Q8PGA3""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
"""Q9RFR9""","""A0A6Q8PGA3""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
…,…,…,…,…,…,…
"""Q3Y0J6""","""O43598""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
"""A0A829F2V1""","""O43598""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null
"""A0A6P1GFG5""","""Q17R31""","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…","""/home/juliandeanmoran/___git_r…",null,null


In [21]:
# ============================================================
#                             Out
# ============================================================

df_structures_aligned.write_csv(
    f"{args['out_dir']}/pymol_alignment_performance_{PM_ALIGN_METHOD}.tsv",
    separator="\t",
    include_header=True
)

In [24]:
rmsd, aligned_atoms = pymol_align_pair(
    pdb_query=df_structure_files[0, col_query_struct],
    pdb_target=df_structure_files[0, col_target_struct],
    pymol_align_method="super",
    out_path_structure="./tmp.pse"
)

INFO:__main__:For /home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/A0A1Y6A5H7_AF.pdb,/home/juliandeanmoran/___git_repos/BISSH/results/test_pymol_align/alphafold_structures/A0A6Q8PGV8_AF.pdb, wrote alignment results to ./tmp.pse.
